In [1]:
import os
os.chdir("/home/ubuntu/work/dahyeon/backend")
os.getcwd()

'/home/ubuntu/work/dahyeon/backend'

In [2]:
import asyncio
import json
import uuid
import time
import random
import requests
import nest_asyncio
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from app.modules.RAG.retriever import (
    full_bm25_with_onboarding,
    full_dense_with_onboarding,
    full_hybrid_with_onboarding,
    chunk_dense_with_onboarding
)
import sys
from app.modules.RAG.anchor_book_pipeline import run_anchor_pipeline
from app.core.config import settings
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

nest_asyncio.apply()

/home/ubuntu/anaconda3/envs/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ubuntu/anaconda3/envs/venv/lib/python3.11/site-packages/elasticsearch/_sync/client/__init__.py:404: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(


In [3]:
REPO_ROOT     = Path("/home/ubuntu/work/dahyeon")
DATASET_DIR   = REPO_ROOT / "evaluation" / "dataset"
SCENARIO_PATH = DATASET_DIR / "scenario_data.json"

with open(SCENARIO_PATH, encoding="utf-8") as f:
    scenarios = json.load(f)

def extract_rag_query(scenario: dict) -> dict:
    """마지막 turn의 rag_query를 꺼내고 query_id(=scenario_id)를 추가한다."""
    rag_query = scenario["turns"][-1]["rag_query"].copy()
    rag_query["query_id"] = scenario["scenario_id"]
    return rag_query

In [4]:
def simplify_item(item, source_name):
    return {
        "isbn":         item.get("isbn"),
        "title":        item.get("title"),
        "author":       item.get("author"),
        "publisher":    item.get("publisher"),
        "publish_date": item.get("publish_date"),
        "page":         item.get("page"),
        "large_cate":   item.get("large_cate"),
        "mid_cate":     item.get("mid_cate"),
        "small_cate":   item.get("small_cate"),
        "book_intro":   item.get("book_intro"),
        "book_index":   item.get("book_index"),
        "review_count": item.get("review_count"),
        "review_text":  item.get("review_text"),
    }


In [5]:
# ============================================================
# Strategy1/2/3 Dense Retrieval 비교
# - embedding model: KURE 고정
# - strategy1: chunk_dense_with_onboarding
# - strategy2, strategy3: full_dense_with_onboarding
# - isbn만 저장
# ============================================================

from pathlib import Path
import json
import pandas as pd
from tqdm import tqdm

retrieval_configs = {
    "strategy1": {
        "func": chunk_dense_with_onboarding,
        "index_name": "books_chunk_strategy1",
        "type": "chunk"
    },
    "strategy2": {
        "func": full_dense_with_onboarding,
        "index_name": "books_chunk_strategy2",
        "type": "full"
    },
    "strategy3": {
        "func": full_dense_with_onboarding,
        "index_name": "books_chunk_strategy3",
        "type": "full"
    }
}

EMBEDDING_MODEL = "kure"
EMBEDDING_FIELD = "embedding_kure"

CANDIDATE_SIZE = 100
TOP_K = 20

all_candidates = []

for scenario in tqdm(scenarios, desc="strategy retrieval"):

    sample_result = extract_rag_query(scenario)

    if sample_result.get("anchor"):
        sample_result = run_anchor_pipeline(sample_result)

    query_id = sample_result["query_id"]

    for strategy_name, config in retrieval_configs.items():

        retrieval_func = config["func"]
        index_name = config["index_name"]
        retrieval_type = config["type"]

        size = CANDIDATE_SIZE if retrieval_type == "chunk" else TOP_K

        re_dense = retrieval_func(
            sample_result,
            size=size,
            index_name=index_name,
            embedding_field=EMBEDDING_FIELD,
            embedding_model=EMBEDDING_MODEL,
            small_category_embeddings=None
        )

        seen_isbns = set()
        rank = 1

        for book in re_dense:
            isbn = str(book["isbn"])

            if isbn in seen_isbns:
                continue

            seen_isbns.add(isbn)

            all_candidates.append({
                "query_id": query_id,
                "embedding_model": EMBEDDING_MODEL,
                "embedding_field": EMBEDDING_FIELD,

                "strategy": strategy_name,
                "retrieval_type": retrieval_type,
                "index_name": index_name,

                "rank": rank,
                "dense_score": book.get("dense_score"),
                "score": book.get("score"),

                "isbn": isbn,
                "title": book.get("title"),
                "author": book.get("author"),
                "publisher": book.get("publisher"),
                "publish_date": book.get("publish_date"),
                "page": book.get("page"),

                "large_cate": book.get("large_cate"),
                "mid_cate": book.get("mid_cate"),
                "small_cate": book.get("small_cate"),

                "book_intro": book.get("book_intro"),
                "book_index": book.get("book_index"),
                "book_index_text": book.get("book_index_text"),

                "review": book.get("review"),
                "review_text": book.get("review_text"),

                "chunk_type": book.get("chunk_type"),
                "chunk_text": book.get("chunk_text"),
            })

            rank += 1

            if rank > TOP_K:
                break

print(f"\n전체 후보 도서: {len(all_candidates):,}")

strategy_compare_df = pd.DataFrame(all_candidates)
strategy_compare_df.head()

OUTPUT_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/chunk_strategy_dense_eval.jsonl")

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for item in all_candidates:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"저장 완료: {OUTPUT_PATH}")
print(f"총 저장 개수: {len(all_candidates):,}")

strategy retrieval: 100%|██████████| 21/21 [03:36<00:00, 10.30s/it]


전체 후보 도서: 1,260
저장 완료: /home/ubuntu/work/dahyeon/evaluation/dataset/chunk_strategy_dense_eval.jsonl
총 저장 개수: 1,260


In [8]:
# =====================================================
# 1. 파일 경로
# =====================================================

from pathlib import Path
import sys
import json
import pandas as pd

GOLDSET_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/goldset_final.json")
DENSE_RESULT_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/chunk_strategy_dense_eval.jsonl")

REPO_ROOT = Path("/home/ubuntu/work/dahyeon")
sys.path.insert(0, str(REPO_ROOT / "evaluation" / "metrics"))

from metrics import hit_at_k, recall_at_k, mrr_at_k, ndcg_at_k

K_VALUES = [20]
RELEVANCE_THRESHOLD = 2

# =====================================================
# 2. goldset 로드
# =====================================================

with GOLDSET_PATH.open("r", encoding="utf-8") as f:
    goldset = json.load(f)

gold_df = pd.DataFrame(goldset)

print("goldset 개수:", len(gold_df))
display(gold_df.head())

# =====================================================
# 3. strategy dense retrieval 결과 로드
# =====================================================

dense_rows = []

with DENSE_RESULT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        dense_rows.append(json.loads(line))

dense_df = pd.DataFrame(dense_rows)

print("dense 결과 개수:", len(dense_df))
display(dense_df.head())

# =====================================================
# 4. query_id별 정답 ISBN 만들기
#    final_grade >= 2 를 relevant로 사용
# =====================================================

relevant_by_query = {}

for qid, g in gold_df.groupby("query_id"):
    relevant_isbns = (
        g[g["final_grade"] >= RELEVANCE_THRESHOLD]["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )

    relevant_by_query[qid] = set(relevant_isbns)

print("query 수:", len(relevant_by_query))
print("relevant 있는 query 수:", sum(1 for v in relevant_by_query.values() if v))

# =====================================================
# 5. eval_data 구성
#    기준: strategy1 / strategy2 / strategy3
# =====================================================

eval_data = {}

for qid in relevant_by_query:
    eval_data[qid] = {
        "relevant_isbns": relevant_by_query[qid]
    }

for (qid, strategy), g in dense_df.groupby(["query_id", "strategy"]):
    ranked = (
        g.sort_values("rank")["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )

    if qid not in eval_data:
        eval_data[qid] = {
            "relevant_isbns": set()
        }

    eval_data[qid][strategy] = ranked

strategies = sorted(dense_df["strategy"].dropna().unique())

print("strategies:", strategies)

# =====================================================
# 6. strategy별 retrieval metric 계산
# =====================================================

def compute_retrieval_metrics(
    eval_data,
    retrievers,
    ks=K_VALUES,
):
    rows = []

    for qid, d in eval_data.items():
        rel = d["relevant_isbns"]

        if not rel:
            continue

        for src in retrievers:
            ranked = d.get(src, [])

            if not ranked:
                continue

            for k in ks:
                rows.append({
                    "query_id": qid,
                    "strategy": src,
                    "k": k,
                    "hit": hit_at_k(rel, ranked, k),
                    "recall": recall_at_k(rel, ranked, k),
                    "mrr": mrr_at_k(rel, ranked, k),
                    "ndcg": ndcg_at_k(rel, ranked, k),
                })

    return pd.DataFrame(rows)

retrieval_df = compute_retrieval_metrics(
    eval_data=eval_data,
    retrievers=strategies,
    ks=K_VALUES,
)

print(f"총 {len(retrieval_df)}개 행 (시나리오 × strategy × K)")
display(retrieval_df.head(9))

goldset 개수: 1678


,isbn,title,author,publisher,publish_date,page,large_cate,mid_cate,small_cate,book_intro,...,relevance_grade,binary_label,label_status,llm_error,final_grade,confidence,grade_j1,grade_j2,grade_j3,llm_raw_output
0,9788926899304,스피노자의 심리철학 - 긍정과 자유를 통해 심리철학을 꿰뚫다,박삼열 저,한국학술정보,2020-05-29,218,[인문],[철학],[서양철학자],"스피노자는 관념론, 유물론 이외에도 질료형상론, 부수현상론, 평행론, 이중측면설 등...",...,1,0,success,None,1,HIGH,1,1,1,NaN
1,9788930622745,들뢰즈의 차이와 반복 입문 - 뉴엄 리더스 가이드,조 휴즈 저 / 황혜령 역,서광사,2014-05-10,320,[인문],"[대학교재, 인문, 철학]","[서양철학자, 철학]","『들뢰즈의 차이와 반복 입문』은 영화와 미술, 문학, 과학 등의 분야에서 널리 주목...",...,0,0,success,None,1,MEDIUM,0,1,1,NaN
2,9788930629157,중국 병법의 지혜,김기동,서광사,1993-11-01,496,[대학교재/전문서적],"[대학교재, 인문, 철학]","[동양철학일반, 철학]",태공 · 손자 · 도가 · 유가 · 묵자 등 고대 병법가 및 손문 · 장개석 · 모...,...,0,0,success,None,0,HIGH,0,0,0,NaN
3,9788934951148,눈물 한 방울 (원본 노트 특별판) - 이어령의 마지막 노트 2019~2022,이어령 저,김영사,2023-02-24,240,"[시/에세이, 인문]",[인문학일반],[인문교양],마지막 3년간 남긴 친필 원화로\n다시 만나는 인간 이어령의 내면 세계\n사유와 영...,...,1,0,success,None,1,HIGH,1,1,1,NaN
4,9788937438363,너와 세상 사이의 싸움에서 - 프란츠 카프카 잠언‧일기집,프란츠 카프카 저/홍성광 역,민음사,2024-05-31,156,[시/에세이],"[나라별 에세이, 테마에세이]","[기타국가에세이, 일기/편지]","프란츠 카프카 서거 100주기 기념 잠언?일기 모음집\n카프카의 사상, 세계관, 종...",...,1,0,success,None,1,HIGH,1,1,1,NaN


dense 결과 개수: 1260


,query_id,embedding_model,embedding_field,strategy,retrieval_type,index_name,rank,dense_score,score,isbn,...,large_cate,mid_cate,small_cate,book_intro,book_index,book_index_text,review,review_text,chunk_type,chunk_text
0,S01,kure,embedding_kure,strategy1,chunk,books_chunk_strategy1,1,NaN,0.839878,9791157063598,...,[인문],[철학],[철학에세이],현실을 바꿔나갈 힘을 얻는 ‘현장의 인문학’\n이 책의 제목에 나오는 ‘하녀’는 권...,"[철학자와 하녀 그리고 별에 관한 이야기, 철학은 지옥에서 하는 것이다, 배움 이전...",철학자와 하녀 그리고 별에 관한 이야기 철학은 지옥에서 하는 것이다 배움 이전에 배...,"{'reader_reaction': '철학책을 재미있게 읽을 수 있으며, 삶의 방식...","철학책을 재미있게 읽을 수 있으며, 삶의 방식과 철학적 사고를 연결해주는 유익한 경...",review,"철학책을 재미있게 읽을 수 있으며, 삶의 방식과 철학적 사고를 연결해주는 유익한 경..."
1,S01,kure,embedding_kure,strategy1,chunk,books_chunk_strategy1,2,NaN,0.823637,9791165347772,...,"[시/에세이, 예술/대중문화]","[나라별 에세이, 미술, 테마에세이]","[교양미술, 그림에세이, 한국에세이]","“오늘도 맑았다가 흐렸다, 그렇게 미치도록 쨍하기를!”\n미술계와 셀럽, 젊은 예술...","[프롤로그_하나의 세계가 열릴 때, 우리가 닮고 싶던 나날들, 너의 이름은, 우리의...",프롤로그_하나의 세계가 열릴 때 우리가 닮고 싶던 나날들 너의 이름은 우리의 세상은...,{'reader_reaction': '자신을 찾아가는 과정을 담담하게 풀어내며 공감...,자신을 찾아가는 과정을 담담하게 풀어내며 공감을 주는 에세이입니다. 자기 자신을 알...,review,자신을 찾아가는 과정을 담담하게 풀어내며 공감을 주는 에세이입니다. 자기 자신을 알...
2,S01,kure,embedding_kure,strategy1,chunk,books_chunk_strategy1,3,NaN,0.716518,9788954688512,...,[시/에세이],"[나라별 에세이, 테마에세이]","[그림에세이, 한국에세이]",15만 팔로워를 사로잡은 감성 인스타툰\n선천적 부끄럼쟁이가 떠난 바깥 세계로의 여...,"[날아온 연필, 펀자이씨툰의 시작, 에세이_그림 그리는 사람입니다, 선천적 부끄럼쟁...",날아온 연필 펀자이씨툰의 시작 에세이_그림 그리는 사람입니다 선천적 부끄럼쟁이 날 ...,{'reader_reaction': '삶의 방향성과 진중함을 담고 있으며 독자에게 ...,삶의 방향성과 진중함을 담고 있으며 독자에게 감동과 영감을 주는 책입니다. 따뜻한 ...,review,삶의 방향성과 진중함을 담고 있으며 독자에게 감동과 영감을 주는 책입니다. 따뜻한 ...
3,S01,kure,embedding_kure,strategy1,chunk,books_chunk_strategy1,4,NaN,0.652802,9791192097695,...,[시/에세이],"[나라별 에세이, 테마에세이]","[그림에세이, 한국에세이]",“최고의 내가 아니어도 괜찮아. 최대한의 나로 살면 되니까.”\n자존감 지킴이 슌이...,"[프롤로그_나를 믿는 것도 재능이 될 수 있을까, 나도 내가 처음이야, 나를 사용하...",프롤로그_나를 믿는 것도 재능이 될 수 있을까 나도 내가 처음이야 나를 사용하는 방...,"{'reader_reaction': '저에게 쉼을 주는 책으로, 답을 찾기 위해 애...","저에게 쉼을 주는 책으로, 답을 찾기 위해 애쓰는 나를 토닥여주는 시간을 제공합니다...",review,"저에게 쉼을 주는 책으로, 답을 찾기 위해 애쓰는 나를 토닥여주는 시간을 제공합니다..."
4,S01,kure,embedding_kure,strategy1,chunk,books_chunk_strategy1,5,NaN,0.648339,9791197001901,...,"[자기계발, 인문]",[철학],[서양철학자],"적나라한 철학자 니체를 실천하라, 그리고 넘어서라!\n강연과 저술 활동으로 대중에게...","[니체처럼 흔들어라, 진지하게 나의 길을 물어라, 오직 나의 두 발로 걸어라]",니체처럼 흔들어라 진지하게 나의 길을 물어라 오직 나의 두 발로 걸어라,{'reader_reaction': '책소개와 인용구를 통해 깊은 인상을 받았으며 ...,책소개와 인용구를 통해 깊은 인상을 받았으며 독서를 통해 지식인이 되기를 희망합니다...,review,책소개와 인용구를 통해 깊은 인상을 받았으며 독서를 통해 지식인이 되기를 희망합니다...


query 수: 21
relevant 있는 query 수: 21
strategies: ['strategy1', 'strategy2', 'strategy3']
총 63개 행 (시나리오 × strategy × K)


,query_id,strategy,k,hit,recall,mrr,ndcg
0,S01,strategy1,20,1,0.166667,0.200000,0.117063
1,S01,strategy2,20,1,0.166667,0.250000,0.130324
2,S01,strategy3,20,1,0.166667,1.000000,0.302602
3,S02,strategy1,20,0,0.000000,0.000000,0.000000
4,S02,strategy2,20,1,0.333333,0.142857,0.156426
5,S02,strategy3,20,0,0.000000,0.000000,0.000000
6,S03,strategy1,20,1,1.000000,0.333333,0.483813
7,S03,strategy2,20,1,1.000000,0.333333,0.483813
8,S03,strategy3,20,1,1.000000,0.500000,0.624051


In [9]:
# =====================================================
# 7. strategy 평균 성능
# =====================================================

summary = (
    retrieval_df
    .groupby(["strategy", "k"])[["hit", "recall", "mrr", "ndcg"]]
    .mean()
    .round(4)
)

print("[strategy 평균 성능]")
print(summary.to_string())

[strategy 평균 성능]
                 hit  recall     mrr    ndcg
strategy  k                                 
strategy1 20  0.9048  0.2342  0.2996  0.2041
strategy2 20  0.9524  0.2631  0.4468  0.2555
strategy3 20  0.8571  0.2078  0.4866  0.2407
